# Lab 5: NumPy Arrays and Gridded Earth Data

        **Week:** Week 5

        **Lab type:** Individual lab

        **Estimated time:** 2 lab periods

        ## Learning objectives

        - Convert tabular grid data to arrays.
- Use masks and array calculations.
- Calculate gradients.
- Make gridded plots.

        ## Earth and environmental motivation

        Gridded Earth data such as elevation and temperature rasters are naturally represented as arrays.

        ## Dataset

        Synthetic gridded elevation and temperature CSV files

        ## Python concepts used

        - NumPy arrays
- Reshape
- Masks
- Gradients
- imshow

## Lab 4 Debrief and Collaborative Debugging (First 10 Minutes)

Open the debrief card from Lab 4. Two to four students or groups will
share a solved problem, an unresolved problem with evidence, or a verification
choice. Work one unresolved problem together, then report to the class.

- 0-5 min: student discussion. Compare cards in small groups and
  debug one unresolved problem together.
- 5-10 min: student reports. Two to four groups report, and the
  class records one reusable lesson.


## Required imports and project paths

Run this cell first. It finds the project root whether the notebook is opened from the
repository root or from a notebook folder.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print(f"Project root: {PROJECT_ROOT}")
print(f"Processed data folder exists: {PROCESSED_DIR.exists()}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

elev = pd.read_csv(PROCESSED_DIR / "synthetic_elevation_grid.csv")
temp = pd.read_csv(PROCESSED_DIR / "synthetic_temperature_grid.csv")
nx = elev["x_km"].nunique()
ny = elev["y_km"].nunique()
elevation = elev["elevation_m"].to_numpy().reshape(ny, nx)
temperature = temp["temperature_c"].to_numpy().reshape(ny, nx)
print(elevation.shape, temperature.shape)

In [ ]:
slope_y, slope_x = np.gradient(elevation)
slope_magnitude = np.sqrt(slope_x**2 + slope_y**2)
cool_high = (temperature < np.percentile(temperature, 25)) & (elevation > np.percentile(elevation, 75))
print("Mean elevation:", elevation.mean())
print("Cool high-elevation cells:", cool_high.sum())

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(elevation, origin="lower", cmap="terrain")
ax.set_title("Synthetic elevation grid")
ax.set_xlabel("Grid column")
ax.set_ylabel("Grid row")
fig.colorbar(image, ax=ax, label="Elevation (m)")
fig.tight_layout()
plt.show()

## Guided coding: slices are subregions

On a gridded array, a slice is a map subregion and a single row or column is
a terrain profile. The grid is 50 x 50, so rows 25 and above are the northern
half.

In [ ]:
north_half = elevation[25:, :]
south_half = elevation[:25, :]
print(f"North-half mean elevation: {north_half.mean():.1f} m")
print(f"South-half mean elevation: {south_half.mean():.1f} m")

xs = np.sort(elev["x_km"].unique())
row_profile = elevation[25, :]
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(xs, row_profile)
ax.set_xlabel("x (km)")
ax.set_ylabel("Elevation (m)")
ax.set_title("Elevation profile along grid row 25")
fig.tight_layout()
plt.show()

## Guided coding: gradients with real spacing

`np.gradient` needs the physical distance between grid cells, otherwise the
slope units are wrong. The spacing comes from the coordinate columns, and the
result is converted to percent slope.

In [ ]:
ys = np.sort(elev["y_km"].unique())
dx_m = (xs[1] - xs[0]) * 1000
dy_m = (ys[1] - ys[0]) * 1000

dz_dy, dz_dx = np.gradient(elevation, dy_m, dx_m)
slope_percent = np.sqrt(dz_dx**2 + dz_dy**2) * 100
print(f"Median slope: {np.median(slope_percent):.2f} percent")

steepest_row, steepest_col = np.unravel_index(np.argmax(slope_percent), slope_percent.shape)
print(f"Steepest cell: row {steepest_row}, column {steepest_col}, "
      f"{slope_percent[steepest_row, steepest_col]:.2f} percent")

## Guided coding: does temperature follow elevation?

Flatten both grids with `.ravel()` and fit a straight line. The slope of that
line is an apparent lapse rate for this synthetic landscape.

In [ ]:
elev_flat = elevation.ravel()
temp_flat = temperature.ravel()
fit_slope, fit_intercept = np.polyfit(elev_flat, temp_flat, 1)
print(f"Fitted lapse rate: {fit_slope * 100:.2f} deg C per 100 m")

fig, ax = plt.subplots(figsize=(6, 4))
ax.scatter(elev_flat, temp_flat, s=4, alpha=0.3, label="Grid cells")
line_x = np.array([elev_flat.min(), elev_flat.max()])
ax.plot(line_x, fit_intercept + fit_slope * line_x, color="crimson", label="Linear fit")
ax.set_xlabel("Elevation (m)")
ax.set_ylabel("Temperature (deg C)")
ax.set_title("Temperature versus elevation")
ax.legend()
fig.tight_layout()
plt.show()

## Try it yourself

Plot the temperature grid and compare it with the elevation grid.

## Graded Checkpoint: Independent Analysis

The guided cells are examples. Complete the task below with your own code; an
unchanged guided notebook does not meet the submission requirement.

Define high and low elevation masks using thresholds you report. Compare mean temperature between the two regions, assert that both masks select at least one cell, and plot the temperature difference or the selected masks.


In [ ]:
# GRADED CHECKPOINT
# Write your code below. Include at least one verification check.


### Scientific Explanation

Replace this text with your interpretation. State what the result means, cite
one piece of numerical or graphical evidence, and name one limitation.


## More practice

1. What percentage of grid cells are BOTH above the median elevation AND
   warmer than the median temperature? Explain why the number is small.
2. Plot an elevation profile along one grid column (a north-south transect)
   with labeled axes.
3. Split the grid into steeper and gentler halves at the median slope and
   compare the mean temperature of the two groups.

## Common mistakes and debugging tips

- Check that `PROCESSED_DIR.exists()` printed `True`.
- Read error messages from the bottom upward.
- Check column names with `df.columns` before selecting a column.
- Keep units in figure labels and written interpretations.
- Re-run earlier cells after changing data-loading or helper-code cells.

## Deliverables checklist

        - [ ] Array shape check
- [ ] One mask-based summary
- [ ] One gridded plot

        ## Short reflection

        What information is easier to see in a gridded plot than in a table?

        ## Rubric summary

        Correctness and completion, readable code, labeled figures, interpretation,
        and reproducibility all matter. Your submitted notebook should run from top to bottom.

## Debrief Card for the Next Lab

Complete this before the next Wednesday meeting. An unresolved problem is a
valid and useful report.

**Goal:** Replace this text with what you were trying to calculate or show.

**Expected result:** Replace this text.

**What happened:** Replace this text with the result, error, or design choice.

**Evidence:** Include an error message, value, figure observation, or tiny test.

**What I tried:** Replace this text.

**Fix or next check:** State what fixed it, or what the class should test next.

**Lesson from a classmate:** Complete this during the next debrief.
